In [ ]:
# %% [0] Geometry, Mesh & Material Configuration  --  BUFFERED / SPLIT-ELECTRODE VARIANT
import warnings
from collections import OrderedDict
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
from shapely.affinity import scale as _scale
from shapely.geometry import Polygon, box
from shapely.ops import unary_union
from skfem import MeshTri
from femwell.mesh import mesh_from_OrderedDict

warnings.filterwarnings("ignore")

# ============================================================
# 1. FIXED PHYSICAL & GEOMETRIC PARAMETERS
# ============================================================
wl = 1.575          # um
wl_m = wl * 1e-6
k0 = 2.0 * np.pi / wl               # 1/um
k0_m = k0 * 1e6                     # 1/m
L_cm = 1.0          # cm
V_bias = 1.0        # V
r33 = 30.8e-12      # m/V
n_guess = 1.8755
num_modes_search = 10               # +2 vs. previous: tighter gap => more hybrid SPP modes near the shift

# --- Vertical stack (um), y = 0 at the BOX / LN interface --------------------
BOX_H = 4.7
TECN = 0.550
WG_H = 0.275
SLAB_H = TECN - WG_H                # 0.275  -> top of the LN slab
WG_TOP = 1.0
ALPHA = 60.0
CLAD_H = 15.0

BUF_H = 0.100                       # NEW: SiO2 buffer decoupling slab from electrodes
EL_BOT_H = 0.500                    # NEW: bottom electrode thickness
EL_TOP_H = 0.500                    # NEW: top electrode thickness

# --- Lateral (um) ------------------------------------------------------------
GAP_BOT = 3.00                      # NEW: bottom-electrode gap
GAP_TOP = 2.50                      # NEW: top-electrode gap (protrudes 0.25 um/side toward the rib)
EL_W = 30.0                         # bottom-electrode width; dissipates the lateral SPP
CAP_W = GAP_BOT                     # NEW: cap widened to exactly fill the bottom gap
CAP_H = 1.40                        # cap height above the slab (see NOTE below)

# NOTE on CAP_H -- the one parameter the spec left open.
#   CAP_H = 1.40 (kept, "all other parameters equal"): the cap is TALLER than the
#     electrode stack, so the top-electrode overhang is embedded in the oxide and a
#     0.25 x 0.30 um block of SiO2 sits above it. Physical (trench-filled metal).
#   CAP_H = BUF_H + EL_BOT_H = 0.60: cap top flush with the bottom electrode, so the
#     overhang sits ON the cap -- this is what the reference sketch literally shows,
#     but it leaves only 0.325 um of oxide over the rib and will change neff/loss.
#   Change this single line to switch; the booleans below handle either case.

basetta = WG_H / np.tan(np.deg2rad(ALPHA))
WG_BOTTOM = WG_TOP + 2.0 * basetta

# --- Derived interface levels & x-stations ----------------------------------
Y_SLAB = SLAB_H                     # top of LN slab            0.275
Y_BUF = Y_SLAB + BUF_H              # top of buffer = el. floor 0.375
Y_EB = Y_BUF + EL_BOT_H             # bottom/top electrode seam 0.875
Y_ET = Y_EB + EL_TOP_H              # top of electrode stack    1.375

XB = GAP_BOT / 2.0                  # 1.50  inner edge of bottom electrode = cap edge
XT = GAP_TOP / 2.0                  # 1.25  inner edge of top electrode
X_OUT = XB + EL_W                   # 31.50 outer edge, shared by both electrode halves
DEV_W = 2.0 * X_OUT + 10.0          # 73.0

# --- Refractive Indices & Permittivities ------------------------------------
ne, no = 2.136842, 2.210268
n_sio2 = 1.438749
eps_au = -120.7 - 11.9j             # Physical gold loss: Im(eps) < 0 in exp(+jwt)

# Skin layer configuration (70 nm strip resolves the 23 nm optical skin depth)
SKIN_T = 0.070
SKIN_SEGMENTS = [(0.0, 5.0, 1.0), (5.0, 10.0, 1.6), (10.0, 20.0, 2.5)]

# Mesh scales (from the convergence study)
h_core = 0.040
h_skin = 0.025
h_buf = 0.030                       # NEW: buffer is only 100 nm thick -> ~3 layers across

MATERIALS = {
    "LN":     {"color": "#2ecc71", "eps_dc": (28.0, 44.0), "eps_opt": (ne**2, no**2, no**2), "desc": "LN Core / Slab"},
    "SiO2":   {"color": "#00ebfc", "eps_dc": (3.75, 3.75), "eps_opt": (n_sio2**2, n_sio2**2, n_sio2**2), "desc": "SiO2 Cap / Buffer / Underclad"},
    "air":    {"color": "#ffffff", "eps_dc": (1.00, 1.00), "eps_opt": (1.00, 1.00, 1.00), "desc": "Air Cladding"},
    "Au_sig": {"color": "#f39c12", "eps_dc": (1.00, 1.00), "eps_opt": (eps_au, eps_au, eps_au), "desc": "Signal Electrode (+1V)"},
    "Au_gnd": {"color": "#e67e22", "eps_dc": (1.00, 1.00), "eps_opt": (eps_au, eps_au, eps_au), "desc": "Ground Electrode (0V)"},
}

# "buf" added; note "buf" never shadows "box" under startswith().
PREFIX_TO_MAT = [
    ("core", "LN"), ("cap", "SiO2"), ("slab", "LN"), ("box", "SiO2"),
    ("buf", "SiO2"), ("clad", "air"), ("elR", "Au_sig"), ("elL", "Au_gnd"),
]


def material_of(region_name):
    base = region_name.split("___")[0]
    for prefix, mat in PREFIX_TO_MAT:
        if base.startswith(prefix):
            return mat
    raise KeyError(f"Unknown material for region: {region_name}")


def mirror(p):
    return _scale(p, xfact=-1.0, yfact=1.0, origin=(0, 0))


# ============================================================
# 2. DOMAIN DISCRETIZATION
#    Returns (polygons, resolutions) together so the two can never drift apart.
#    Every region is built strictly non-overlapping; the asserts below prove it.
# ============================================================
def build_polygons():
    polys, res = OrderedDict(), {}

    # ---- LN rib ------------------------------------------------------------
    core = Polygon([
        (-WG_BOTTOM / 2.0, Y_SLAB),
        (-WG_TOP / 2.0, Y_SLAB + WG_H),
        (WG_TOP / 2.0, Y_SLAB + WG_H),
        (WG_BOTTOM / 2.0, Y_SLAB),
    ])

    # ---- Electrodes: drawn as two blocks, unioned into ONE conductor body --
    # Outer edges are flush at X_OUT; only the inner edge differs (XB vs XT).
    el_bot_r = box(XB, Y_BUF, X_OUT, Y_EB)
    el_top_r = box(XT, Y_EB, X_OUT, Y_ET)
    el_r = unary_union([el_bot_r, el_top_r])          # fused -> single Polygon

    # ---- Gold skin strips on the optically-exposed metal faces -------------
    # (a) underside of the bottom electrode, graded in x  (the dominant Au/SiO2 SPP face)
    skins_r = []
    for (a, b, _f) in SKIN_SEGMENTS:
        a_c, b_c = min(a, EL_W), min(b, EL_W)
        if b_c <= a_c:
            continue
        skins_r.append(box(XB + a_c, Y_BUF, XB + b_c, Y_BUF + SKIN_T))
    # (b) inner sidewall of the bottom electrode, merged into segment 0
    skins_r[0] = unary_union([skins_r[0], box(XB, Y_BUF, XB + SKIN_T, Y_EB)])
    # (c) inner sidewall of the TOP electrode + underside of its 0.25 um overhang.
    #     This wraps the re-entrant metal corner nearest the rib - the new loss hot spot.
    skin_top_r = unary_union([
        box(XT, Y_EB, XT + SKIN_T, Y_ET),
        box(XT, Y_EB, XB, Y_EB + SKIN_T),
    ]).difference(unary_union(skins_r))
    bulk_r = el_r.difference(unary_union(skins_r + [skin_top_r]))
    skin_len = min(SKIN_SEGMENTS[-1][1], EL_W)

    # ---- Oxide: cap (over the gap) + 100 nm buffer (everywhere else) -------
    # Booleans are metal-wins, so this stays valid for any CAP_H.
    cap = box(-XB, Y_SLAB, XB, Y_SLAB + CAP_H).difference(
        unary_union([core, el_r, mirror(el_r)])
    )
    buf_full = box(-DEV_W / 2.0, Y_SLAB, DEV_W / 2.0, Y_BUF)
    buf_segs = [box(XB + a, Y_SLAB, XB + b, Y_BUF) for (a, b, _f) in SKIN_SEGMENTS]
    buf_far = buf_full.difference(unary_union(
        buf_segs + [mirror(s) for s in buf_segs] + [box(-XB, Y_SLAB, XB, Y_BUF)]
    ))

    # ---- Slab / BOX / cladding --------------------------------------------
    slab_near = box(-XB - skin_len, 0.0, XB + skin_len, Y_SLAB)
    slab_far = box(-DEV_W / 2.0, 0.0, DEV_W / 2.0, Y_SLAB).difference(slab_near)

    box_near = box(-XB - 1.0, -1.5, XB + 1.0, 0.0)
    box_far = box(-DEV_W / 2.0, -BOX_H, DEV_W / 2.0, 0.0).difference(box_near)

    solids = unary_union([core, cap, el_r, mirror(el_r), buf_full])
    clad = box(-DEV_W / 2.0, Y_SLAB, DEV_W / 2.0, Y_SLAB + CLAD_H).difference(solids)
    clad_gap = clad.intersection(box(-XB - 0.5, Y_SLAB, XB + 0.5, Y_SLAB + 2.5))
    clad_far = clad.difference(clad_gap)

    # ---- Register regions + their mesh sizes -------------------------------
    def add(name, poly, resolution, distance):
        polys[name] = poly
        res[name] = {"resolution": resolution, "distance": distance}

    add("core", core, h_core, 0.30)
    add("cap", cap, 1.5 * h_core, 0.30)

    for i, (s, (_a, _b, f)) in enumerate(zip(skins_r, SKIN_SEGMENTS)):
        add(f"elR_bskin{i}", s, f * h_skin, 0.20)
        add(f"elL_bskin{i}", mirror(s), f * h_skin, 0.20)
    add("elR_tskin", skin_top_r, h_skin, 0.20)
    add("elL_tskin", mirror(skin_top_r), h_skin, 0.20)
    add("elR_bulk", bulk_r, 0.300, 0.50)
    add("elL_bulk", mirror(bulk_r), 0.300, 0.50)

    for i, (s, (_a, _b, f)) in enumerate(zip(buf_segs, SKIN_SEGMENTS)):
        add(f"bufR_{i}", s, f * h_buf, 0.20)
        add(f"bufL_{i}", mirror(s), f * h_buf, 0.20)
    add("buf_far", buf_far, 0.120, 0.50)

    add("slab_near", slab_near, 0.050, 0.30)
    add("slab_far", slab_far, 0.150, 0.50)
    add("box_near", box_near, 0.080, 0.50)
    add("box_far", box_far, 0.800, 1.00)
    add("clad_gap", clad_gap, 2.0 * h_core, 0.50)
    add("clad_far", clad_far, 0.800, 1.00)

    return polys, res


polygons, resolutions = build_polygons()

# ============================================================
# 3. GEOMETRY SANITY CHECKS (cheap; protects you when editing parameters)
# ============================================================
_names = list(polygons)
for _n, _p in polygons.items():
    assert _p.is_valid and not _p.is_empty and _p.area > 1e-9, f"degenerate region: {_n}"
for _i in range(len(_names)):
    for _j in range(_i + 1, len(_names)):
        _ov = polygons[_names[_i]].intersection(polygons[_names[_j]]).area
        assert _ov < 1e-10, f"regions overlap: {_names[_i]} & {_names[_j]} ({_ov:.2e})"
_tot = sum(p.area for p in polygons.values())
_full = DEV_W * (BOX_H + Y_SLAB + CLAD_H)
assert abs(_tot - _full) < 1e-8, f"tiling gap/overlap: {_tot:.9f} vs {_full:.9f}"
assert set(resolutions) == set(polygons)
assert WG_BOTTOM / 2.0 < XB, "rib base is wider than the cap / bottom gap"
_sig_body = unary_union([p for n, p in polygons.items() if n.startswith("elR")])
assert _sig_body.geom_type == "Polygon", "signal electrode is not a single fused body"
print(f"[geometry OK] {len(polygons)} regions tile {_full:.4f} um^2 exactly; "
      f"electrode stack fused, area = {_sig_body.area:.4f} um^2")

# ============================================================
# 4. MESH
# ============================================================
raw_mesh = mesh_from_OrderedDict(
    polygons,
    resolutions=resolutions,
    default_resolution_min=0.5 * h_skin,   # must stay BELOW the finest requested size
    default_resolution_max=0.6,            # matches the converged diagnostic
)
skfem_mesh = MeshTri(raw_mesh.points[:, :2].T, raw_mesh.cells_dict["triangle"].T)
tri_subdomains = raw_mesh.cell_data_dict["gmsh:physical"]["triangle"]
name_to_id = {
    name: data[0] if hasattr(data, "__getitem__") else data
    for name, data in raw_mesh.field_data.items()
}

mat_of_el = np.empty(skfem_mesh.nelements, dtype=object)
for raw_name, s_id in name_to_id.items():
    if raw_name.split("___")[0] in polygons:
        mat_of_el[tri_subdomains == s_id] = material_of(raw_name)

assert not np.any(mat_of_el == None), (  # noqa: E711
    f"{int(np.sum(mat_of_el == None))} elements unassigned - a region name escaped PREFIX_TO_MAT"  # noqa: E711
)
print(f"[mesh OK] {skfem_mesh.nelements} triangles, {skfem_mesh.nvertices} vertices")
for _m in MATERIALS:
    print(f"    {_m:<8} {int(np.sum(mat_of_el == _m)):>7d} elements")

print("=" * 84)
print(f"{'CANONICAL MATERIAL':<32} | {'DC EPS (x, y)':<16} | {'OPTICAL EPS (xx, yy, zz)':<28}")
print("-" * 84)
for key, data in MATERIALS.items():
    dc_s = f"({data['eps_dc'][0]:.1f}, {data['eps_dc'][1]:.1f})"
    opt_val = data["eps_opt"][0]
    opt_s = f"({opt_val.real:.1f}{opt_val.imag:+.1f}j)" if np.iscomplex(opt_val) else f"({opt_val.real:.3f})"
    print(f"{data['desc']:<32} | {dc_s:<16} | {opt_s:<28}")
print("=" * 84)

# ============================================================
# 5. CROSS-SECTION VISUALIZATION
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
for ax, (xlim, ylim, ttl) in zip(axes, [
    ((-8.0, 8.0), (-1.5, 2.5), "Full cross-section"),
    ((-2.2, 2.2), (0.1, 1.8), "Gap detail: buffer, cap & split electrodes"),
]):
    legend_patches, seen = [], set()
    for raw_name, s_id in name_to_id.items():
        if raw_name.split("___")[0] not in polygons:
            continue
        mat_key = material_of(raw_name)
        col = MATERIALS[mat_key]["color"]
        elem_idx = np.where(tri_subdomains == s_id)[0]
        if elem_idx.size == 0:
            continue
        sub_mesh = MeshTri(skfem_mesh.p, skfem_mesh.t[:, elem_idx])
        sub_mesh.plot(np.zeros(sub_mesh.nelements), ax=ax, shading="flat",
                      cmap=plt.matplotlib.colors.ListedColormap([col]))
        sub_mesh.draw(ax=ax, color="black", lw=0.15)
        if mat_key not in seen:
            legend_patches.append(mpatches.Patch(color=col, label=f"{mat_key}: {MATERIALS[mat_key]['desc']}"))
            seen.add(mat_key)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_title(ttl, fontsize=10)
    ax.set_xlabel(r"$x$ ($\mu$m)")
    ax.set_ylabel(r"$y$ ($\mu$m)")
    ax.set_aspect("equal")
axes[0].legend(handles=legend_patches, loc="upper right", fontsize=7, framealpha=0.9)
fig.suptitle(
    rf"TFLN rib + {BUF_H*1e3:.0f} nm SiO$_2$ buffer, split electrodes "
    rf"(gap$_{{bot}}$={GAP_BOT}, gap$_{{top}}$={GAP_TOP} $\mu$m) | "
    rf"$h_{{core}}$={h_core}, $h_{{skin}}$={h_skin}, $h_{{buf}}$={h_buf} $\mu$m"
)
plt.tight_layout()
plt.show()


In [ ]:
# %% [1] Anisotropic DC Electrostatics
from skfem import Basis, ElementTriP0, ElementTriP1, BilinearForm, asm, condense, solve
from skfem.helpers import grad

basis_dc = Basis(skfem_mesh, ElementTriP1())
basis_p0 = Basis(skfem_mesh, ElementTriP0())

eps_dc_x = np.array([MATERIALS[m]["eps_dc"][0] for m in mat_of_el], dtype=float)
eps_dc_y = np.array([MATERIALS[m]["eps_dc"][1] for m in mat_of_el], dtype=float)


@BilinearForm
def laplace_aniso(u, v, w):
    return w.eps_x * grad(u)[0] * grad(v)[0] + w.eps_y * grad(u)[1] * grad(v)[1]


K = asm(
    laplace_aniso,
    basis_dc,
    eps_x=basis_p0.interpolate(eps_dc_x),
    eps_y=basis_p0.interpolate(eps_dc_y),
)

# ------------------------------------------------------------------
# ELECTRODE BOUNDARY CONDITION
# ------------------------------------------------------------------
# get_dofs(elements=...) is VOLUMETRIC, not perimeter-based: internally it does
# np.unique(topo.t[:, elements]), i.e. every node touched by the selected elements
# (interior nodes included). So the whole conductor body is pinned to one potential.
#
# Consequences for the split (bottom + top) electrode:
#   * no perimeter is ever extracted, so there is no "wrong perimeter" to pick;
#   * the bottom/top seam nodes are pinned to the SAME value from both blocks, so
#     no spurious internal Dirichlet surface and no artificial charge sheet appear;
#   * the effective conductor surface is automatically the OUTER boundary of the
#     fused body, which is exactly what we want.
# The only real requirement is that every gold sub-region map to the same material
# tag -- guaranteed by the elR_* / elL_* naming + PREFIX_TO_MAT. Verified below.
# ------------------------------------------------------------------
sig_regions = sorted(n for n in polygons if n.startswith("elR"))
gnd_regions = sorted(n for n in polygons if n.startswith("elL"))
print(f"Signal sub-regions ({len(sig_regions)}): {sig_regions}")
print(f"Ground sub-regions ({len(gnd_regions)}): {gnd_regions}")

sig_elements = np.where(mat_of_el == "Au_sig")[0]
gnd_elements = np.where(mat_of_el == "Au_gnd")[0]
assert sig_elements.size and gnd_elements.size, "an electrode has no elements"

dofs_signal = basis_dc.get_dofs(elements=sig_elements).all()
dofs_ground = basis_dc.get_dofs(elements=gnd_elements).all()
assert np.intersect1d(dofs_signal, dofs_ground).size == 0, "signal and ground electrodes touch"
dofs_dirichlet = np.unique(np.concatenate([dofs_signal, dofs_ground]))

u_dirichlet = np.zeros(basis_dc.N)
u_dirichlet[dofs_signal] = V_bias
u_dirichlet[dofs_ground] = 0.0

K_c, f_c, u_c, I = condense(K, np.zeros(basis_dc.N), x=u_dirichlet, D=dofs_dirichlet)
potential_nodes = u_c.copy()
potential_nodes[I] = solve(K_c, f_c)

grad_V = basis_dc.interpolate(potential_nodes).grad
Ex_dc = -np.mean(grad_V[0], axis=-1) if grad_V.ndim == 3 else -grad_V[0]
Ey_dc = -np.mean(grad_V[1], axis=-1) if grad_V.ndim == 3 else -grad_V[1]

# --- Proof that the fused electrode really is equipotential --------------------
metal = np.isin(mat_of_el, ["Au_sig", "Au_gnd"])
E_in_metal = float(np.max(np.hypot(Ex_dc[metal], Ey_dc[metal])))
E_nominal = V_bias / GAP_BOT
print(f"Pinned DOFs: {dofs_signal.size} signal + {dofs_ground.size} ground "
      f"= {dofs_dirichlet.size} of {basis_dc.N}")
print(f"max |E_dc| inside metal = {E_in_metal:.3e} V/um  "
      f"({E_in_metal / E_nominal:.1e} x nominal V/gap)  -> body is equipotential, "
      f"bottom/top seam is transparent")

# ------------------------------------------------------------------
# Diagnostic plots
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(17, 4.0))

ax0 = axes[0]
electrode_field = np.zeros(skfem_mesh.nelements)
electrode_field[mat_of_el == "Au_sig"] = 1.0
electrode_field[mat_of_el == "Au_gnd"] = -1.0
skfem_mesh.plot(electrode_field, ax=ax0, shading="flat",
                cmap=plt.matplotlib.colors.ListedColormap(["#2c3e50", "#ecf0f1", "#e67e22"]))
ax0.text(XB + 0.6, Y_EB + 0.15, "SIGNAL\n(+1V)", color="white", fontweight="bold", fontsize=8)
ax0.text(-XB - 3.4, Y_EB + 0.15, "GROUND\n(0V)", color="white", fontweight="bold", fontsize=8)
ax0.set_title("Electrode assignment (both halves = one body)", fontsize=10)
ax0.set_xlim([-8.0, 8.0])
ax0.set_ylim([-0.5, 2.0])
ax0.set_aspect("equal")

ax1 = axes[1]
skfem_mesh.plot(potential_nodes, ax=ax1, shading="gouraud", cmap="coolwarm", colorbar=True)
ax1.set_title(r"Anisotropic DC potential $V(x,y)$ [V]", fontsize=10)
ax1.set_xlim([-6.0, 6.0])
ax1.set_ylim([-0.5, 2.0])
ax1.set_aspect("equal")

ax2 = axes[2]
skfem_mesh.plot(Ex_dc, ax=ax2, shading="flat", cmap="RdBu_r", colorbar=True)
ax2.set_title(r"$E_x^{\,DC}$ [V/$\mu$m] - buffer & gap detail", fontsize=10)
ax2.set_xlim([-2.6, 2.6])
ax2.set_ylim([0.0, 1.8])
ax2.set_aspect("equal")

plt.tight_layout()
plt.show()


In [ ]:
# %% [2] Optical Eigensolver (N1 x P1 Elements)
import scipy.constants
from skfem import ElementTriN1, ElementTriP0, ElementTriP1, BilinearForm, solve, condense
from skfem.helpers import curl, dot, grad
from skfem.utils import solver_eigen_scipy
from femwell.maxwell.waveguide import Mode, Modes, calculate_hfield, calculate_overlap


def compute_modes_anisotropic(
    basis_epsilon_r,
    eps_xx,
    eps_yy,
    eps_zz,
    wavelength,
    num_modes=8,
    n_guess=1.8755,
):
    c0 = scipy.constants.speed_of_light
    k0_val = 2.0 * np.pi / wavelength
    element = ElementTriN1() * ElementTriP1()
    basis = basis_epsilon_r.with_element(element)
    basis_eps = basis.with_element(basis_epsilon_r.elem)

    @BilinearForm(dtype=complex)
    def aform(e_t, e_z, v_t, v_z, w):
        return (
            curl(e_t) * curl(v_t) / k0_val**2
            - (w.eps_xx * e_t[0] * v_t[0] + w.eps_yy * e_t[1] * v_t[1])
            + dot(grad(e_z), v_t)
            + (w.eps_xx * e_t[0] * grad(v_z)[0] + w.eps_yy * e_t[1] * grad(v_z)[1])
            - w.eps_zz * e_z * v_z * k0_val**2
        )

    @BilinearForm(dtype=complex)
    def bform(e_t, e_z, v_t, v_z, w):
        return -dot(e_t, v_t) / k0_val**2

    A = aform.assemble(
        basis,
        eps_xx=basis_eps.interpolate(eps_xx),
        eps_yy=basis_eps.interpolate(eps_yy),
        eps_zz=basis_eps.interpolate(eps_zz),
    )
    B = bform.assemble(basis)

    bnd_dofs = basis.get_dofs(facets=skfem_mesh.boundary_facets())
    lams, xs = solve(
        *condense(-A, -B, D=bnd_dofs, x=basis.zeros(dtype=complex)),
        solver=solver_eigen_scipy(k=num_modes, sigma=k0_val**2 * n_guess**2),
    )

    xs[basis.split_indices()[1], :] /= 1j * np.sqrt(lams[np.newaxis, :] / k0_val**4)

    hs = []
    omega = k0_val * (c0 * 1e6)
    for i, lam in enumerate(lams):
        H = calculate_hfield(basis, xs[:, i], np.sqrt(lam), omega=omega)
        power = calculate_overlap(basis, xs[:, i], H, basis, xs[:, i], H)
        xs[:, i] /= np.sqrt(power)
        H /= np.sqrt(power)
        hs.append(H)

    return Modes(
        modes=[
            Mode(
                frequency=(c0 * 1e6) / wavelength,
                k=np.sqrt(lams[i]),
                basis_epsilon_r=basis_epsilon_r,
                epsilon_r=eps_xx,
                basis=basis,
                E=xs[:, i],
                H=hs[i],
            )
            for i in range(num_modes)
        ]
    )


eps_xx = np.array([MATERIALS[m]["eps_opt"][0] for m in mat_of_el], dtype=complex)
eps_yy = np.array([MATERIALS[m]["eps_opt"][1] for m in mat_of_el], dtype=complex)
eps_zz = np.array([MATERIALS[m]["eps_opt"][2] for m in mat_of_el], dtype=complex)

# intorder=4 is REQUIRED, not cosmetic.
#   with_element() inherits the quadrature rule of the basis it is called on.
#   Basis(mesh, ElementTriP0()) defaults to a 3-point rule, so the N1 x P1
#   eigenproblem would be assembled with 3 points, while skfem's native rule for
#   N1 x P1 is 6 points (intorder 4). That under-integrates the curl-curl and mass
#   forms; it bites hardest on the thin, high-aspect-ratio buffer/skin elements.
basis_epsilon_r = Basis(skfem_mesh, ElementTriP0(), intorder=4)

import time
t_eig0 = time.time()
modes = compute_modes_anisotropic(
    basis_epsilon_r,
    eps_xx,
    eps_yy,
    eps_zz,
    wavelength=wl,
    num_modes=num_modes_search,
    n_guess=n_guess,
)
print(f"Eigensolver complete in {time.time() - t_eig0:.1f}s: computed {len(modes)} optical modes around n_guess = {n_guess}.")


In [ ]:
# %% [3] High-Precision Diagnostics & Mode Gallery
from skfem import Functional

core_ids = [s_id for name, s_id in name_to_id.items() if name.split("___")[0] == "core"]
is_core = np.zeros(skfem_mesh.nelements, dtype=float)
is_core[np.isin(tri_subdomains, core_ids)] = 1.0
is_core_field = basis_p0.interpolate(is_core)

@Functional(dtype=complex)
def _px(w):
    return np.abs(w["E"][0][0]) ** 2

@Functional(dtype=complex)
def _ptot(w):
    return (
        np.abs(w["E"][0][0]) ** 2
        + np.abs(w["E"][0][1]) ** 2
        + np.abs(w["E"][1]) ** 2
    )

@Functional(dtype=complex)
def _p_core(w):
    return w["is_core"] * (
        np.abs(w["E"][0][0]) ** 2
        + np.abs(w["E"][0][1]) ** 2
        + np.abs(w["E"][1]) ** 2
    )

@Functional(dtype=complex)
def _ex_core_signed(w):
    return w["is_core"] * np.real(w["E"][0][0])

@Functional(dtype=complex)
def _ex_core_abs(w):
    return w["is_core"] * np.abs(np.real(w["E"][0][0]))

mode_diagnostics = []

print("=" * 82)
print(f"{'Idx':<4} | {'Re(neff)':<10} | {'Im(neff)':<12} | {'TE %':<8} | {'Core Pwr %':<10} | {'Unipolarity':<12}")
print("-" * 82)

for idx, mode in enumerate(modes):
    E_interp = mode.basis.interpolate(mode.E)

    P_x = np.real(_px.assemble(mode.basis, E=E_interp))
    P_tot = np.real(_ptot.assemble(mode.basis, E=E_interp))
    P_core = np.real(_p_core.assemble(mode.basis, E=E_interp, is_core=is_core_field))

    te_ratio = P_x / P_tot
    core_confinement = P_core / P_tot

    signed = np.real(_ex_core_signed.assemble(mode.basis, E=E_interp, is_core=is_core_field))
    absval = np.real(_ex_core_abs.assemble(mode.basis, E=E_interp, is_core=is_core_field))

    unipolarity = 0.0000 if absval < 1e-12 else np.abs(signed) / absval

    neff_val = complex(mode.k / k0)

    mode_diagnostics.append({
        "index": idx,
        "mode": mode,
        "neff": neff_val,
        "te_ratio": te_ratio,
        "core_conf": core_confinement,
        "unipolarity": unipolarity,
    })

    print(
        f"{idx:<4} | "
        f"{neff_val.real:<10.5f} | "
        f"{neff_val.imag:<12.2e} | "
        f"{te_ratio * 100:<7.2f}% | "
        f"{core_confinement * 100:<9.2f}% | "
        f"{unipolarity:<12.4f}"
    )

print("=" * 82)

# Full Mode Gallery Plot
cols = 4
rows = int(np.ceil(len(modes) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(16, 3.4 * rows))
axes = np.array(axes).flatten()

for idx, diag in enumerate(mode_diagnostics):
    ax = axes[idx]
    E_interp = diag["mode"].basis.interpolate(diag["mode"].E)
    Ex_elem = np.mean(np.abs(E_interp[0].value[0]) ** 2, axis=-1)

    skfem_mesh.plot(Ex_elem, ax=ax, shading="flat", cmap="inferno")
    title_str = (
        f"Mode {idx}: n={diag['neff'].real:.4f}\n"
        f"TE={diag['te_ratio']*100:.1f}% | Conf={diag['core_conf']*100:.1f}%\n"
        f"Uni={diag['unipolarity']:.4f}"
    )
    ax.set_title(title_str, fontsize=9)
    ax.set_xlim([-4.0, 4.0])
    ax.set_ylim([-0.5, 1.8])   # taller window: cap now reaches y=1.675
    ax.set_aspect("equal")

for j in range(len(modes), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# %% [4] Overlap Integral, Vpi*L & Optical Attenuation
from skfem import Functional

# Fundamental TE mode identification (highest index matching modal filters)
guided_cands = [
    d for d in mode_diagnostics
    if d["te_ratio"] >= 0.75 and d["core_conf"] >= 0.20 and d["unipolarity"] >= 0.85
]

if not guided_cands:
    raise RuntimeError("Fundamental TE rib mode not found. Check n_guess or eigensolver outputs.")

guided_cands.sort(key=lambda x: x["neff"].real, reverse=True)
fund_diag = guided_cands[0]
chosen_idx = fund_diag["index"]
fund_mode = fund_diag["mode"]
n_eff = fund_diag["neff"]

# 1. Optical Attenuation (Ohmic Metal Loss matching COMSOL Att_Opt)
att_opt_dB_m = np.abs(20.0 * np.log10(np.e) * k0_m * np.imag(n_eff))
att_opt_dB_cm = att_opt_dB_m / 100.0

# 2. Electro-Optic Overlap Integral (Gamma)
ln_mask = (mat_of_el == "LN")
Ex_dc_ln = np.zeros(skfem_mesh.nelements, dtype=float)
Ex_dc_ln[ln_mask] = Ex_dc[ln_mask]
Ex_dc_ln_field = basis_p0.interpolate(Ex_dc_ln)

E_fund_interp = fund_mode.basis.interpolate(fund_mode.E)

@Functional(dtype=complex)
def _num(w): 
    return w["Ex_dc_ln"] * np.abs(w["E"][0][0]) ** 2

@Functional(dtype=complex)
def _den(w): 
    return np.abs(w["E"][0][0]) ** 2

num = np.real(_num.assemble(fund_mode.basis, E=E_fund_interp, Ex_dc_ln=Ex_dc_ln_field))
den = np.real(_den.assemble(fund_mode.basis, E=E_fund_interp))

# GAP_EO only normalises the reported Gamma: it appears in both the overlap
# definition and in Vpi*L, so it cancels exactly and does NOT affect Vpi*L.
# GAP_BOT is the honest choice (it is the gap the DC field actually drops across).
GAP_EO = GAP_BOT
overlap = (GAP_EO / V_bias) * (num / den)
Vpi_L_m = (GAP_EO * 1e-6 * wl_m) / (r33 * np.abs(overlap) * (ne**3) * 2.0)
Vpi_L_Vcm = Vpi_L_m * 100.0
Vpi = Vpi_L_Vcm / L_cm

# 3. Formatted Simulation Summary
print("=" * 55)
print("             ELECTRO-OPTIC SIMULATION RESULTS")
print("=" * 55)
print(f"Selected Mode Index:         {chosen_idx}")
print(f"Effective Index (neff):      {n_eff.real:.6f} + {n_eff.imag:.2e}j")
print(f"Optical Attenuation:         {att_opt_dB_cm:.4e} dB/cm  ({att_opt_dB_m:.4e} dB/m)")
print(f"Overlap Integral (Gamma):    {overlap:.4f}   (normalised to GAP_EO = {GAP_EO} um)")
print(f"|Vpi * L|:                   {Vpi_L_Vcm:.4f} V*cm")
print(f"|Vpi| (for L = {L_cm} cm):        {Vpi:.4f} V")
print("=" * 55)